# Modul 5 — Praktiki iş: Kompüter görməsində neyron şəbəkələr
### CIFAR-10 üzərində sadə modeldən ResNet-ə qədər

Bu praktikada modulun bütün mövzularını bir layihədə birləşdiririk:

| № | Mövzu | Bu notebook-da harada |
|---|-------|---------------------|
| 1 | Təsvirlərin təsnifatı tapşırığı | Bölmə 1 — CIFAR-10 ilə tanışlıq |
| 2 | Konvolyusiya, filtrlər, konvolyusiya qatları | Bölmə 2 — konvolyusiyanı əl ilə yazırıq |
| 3 | Çoxsinifli təsnifat | Bölmə 3 — Softmax və Cross-Entropy |
| 4 | CNN qurmaq | Bölmə 4 — MLP vs CNN, LeNet-5 |
| 5 | Batching, dərin şəbəkələrin problemləri, Batch Normalization | Bölmə 5 və 6 |
| 6 | Arxitekturalar: LeNet, AlexNet, ImageNet, ResNet | Bölmə 4, 7, 8 |
| 7 | Qradiyent normallaşdırması, Gradient clipping, LayerNorm | Bölmə 9 və 10 |

Sonda bütün modelləri müqayisə edəcək, ən yaxşı modelin səhvlərini təhlil edəcək və şəbəkənin "nə gördüyünə" baxacağıq.

> **Vacib:** Colab-da GPU-nu yandırın: **Runtime → Change runtime type → T4 GPU**.
> GPU ilə bütün notebook təxminən 25–40 dəqiqəyə icra olunur.

**Necə işləmək lazımdır:** hər xanadan əvvəl izahı oxuyun, kodu icra edin, nəticəyə baxın və 🤔 işarəli suallara cavab verməyə çalışın.

## 0. Hazırlıq

In [1]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

keras.utils.set_random_seed(42)   # nəticələrin təkrarlana bilməsi üçün

TensorFlow: 2.21.0
Keras: 3.15.1
GPU: []


In [2]:
# Eksperimentlərin əsas parametrləri — vaxtınız azdırsa, epoxaları azaldın
EPOCHS = 15          # əsas modellər üçün epoxa sayı
SHORT_EPOCHS = 5     # qısa eksperimentlər üçün
BATCH_SIZE = 128

## 1. Təsvirlərin təsnifatı tapşırığı: CIFAR-10

**Təsnifat (classification)** — modelə təsvir veririk, o isə təsvirin hansı sinfə aid olduğunu deyir.

**CIFAR-10** datasetı:
- 60 000 rəngli şəkil, ölçüsü **32×32×3** (en × hündürlük × RGB kanalları);
- 10 sinif: təyyarə, avtomobil, quş, pişik, maral, it, qurbağa, at, gəmi, yük maşını;
- 50 000 öyrətmə + 10 000 test şəkli.

MNIST-dən fərqi: şəkillər rənglidir, obyektlər müxtəlif bucaqdan, müxtəlif fonda çəkilib — tapşırıq xeyli çətindir.

In [3]:
(x_train_full, y_train_full), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Etiketlər (N, 1) formasındadır — onları (N,) formasına salırıq
y_train_full = y_train_full.flatten()
y_test = y_test.flatten()

class_names = ["təyyarə", "avtomobil", "quş", "pişik", "maral",
               "it", "qurbağa", "at", "gəmi", "yük maşını"]

print("x_train_full:", x_train_full.shape, x_train_full.dtype)
print("y_train_full:", y_train_full.shape)
print("x_test:", x_test.shape)
print("Piksel dəyərlərinin diapazonu:", x_train_full.min(), "-", x_train_full.max())

x_train_full: (50000, 32, 32, 3) uint8
y_train_full: (50000,)
x_test: (10000, 32, 32, 3)
Piksel dəyərlərinin diapazonu: 0 - 255


In [4]:
plt.figure(figsize=(12, 5))
for i in range(20):
    plt.subplot(2, 10, i + 1)
    plt.imshow(x_train_full[i])
    plt.title(class_names[y_train_full[i]], fontsize=9)
    plt.axis("off")
plt.suptitle("CIFAR-10 nümunələri")
plt.show()

/var/folders/9g/p_yhbqws45xfhb8zmmg367j00000gn/T/ipykernel_11587/2232906447.py:8: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
# Siniflər balanslıdırmı?
counts = np.bincount(y_train_full)
plt.figure(figsize=(10, 3))
plt.bar(class_names, counts)
plt.title("Öyrətmə dəstində siniflərin sayı")
plt.xticks(rotation=45)
plt.show()

/var/folders/9g/p_yhbqws45xfhb8zmmg367j00000gn/T/ipykernel_11587/1583953148.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Məlumatların hazırlanması

1. **Normallaşdırma:** pikselləri `[0, 255]` aralığından `[0, 1]` aralığına salırıq — şəbəkə kiçik ədədlərlə daha stabil öyrənir.
2. **Doğrulama dəsti (validation):** öyrətmə dəstindən 5 000 şəkli ayırırıq. Modeli öyrədərkən keyfiyyəti bu dəstdə izləyirik, test dəstinə isə yalnız sonda baxırıq.

In [6]:
x_train_full = x_train_full.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train, x_val = x_train_full[:45000], x_train_full[45000:]
y_train, y_val = y_train_full[:45000], y_train_full[45000:]

print("Öyrətmə:", x_train.shape, " Doğrulama:", x_val.shape, " Test:", x_test.shape)

Öyrətmə: (45000, 32, 32, 3)  Doğrulama: (5000, 32, 32, 3)  Test: (10000, 32, 32, 3)


## 2. Konvolyusiya və filtrlər — "əl ilə"

**Konvolyusiya** əməliyyatında kiçik bir matris — **filtr (kernel, nüvə)** — şəkil üzərində sürüşdürülür. Hər mövqedə filtrin elementləri altındakı piksellərə vurulur və cəmlənir:

$$
\text{out}[i, j] = \sum_{m}\sum_{n} \text{image}[i+m,\; j+n] \cdot \text{kernel}[m, n]
$$

Nəticədə alınan matrisə **xüsusiyyət xəritəsi (feature map)** deyilir.

Əsas parametrlər:
- **kernel size** — filtrin ölçüsü (məs. 3×3);
- **stride** — filtrin addımı;
- **padding** — kənarlara sıfırların əlavə olunması.

Çıxışın ölçüsü:
$$
W_{out} = \frac{W - K + 2P}{S} + 1
$$

In [7]:
def conv2d_manual(image, kernel, stride=1, padding=0):
    # Bir kanallı şəkil üçün sadə 2D konvolyusiya
    if padding > 0:
        image = np.pad(image, padding, mode="constant")
    kh, kw = kernel.shape
    H, W = image.shape
    out_h = (H - kh) // stride + 1
    out_w = (W - kw) // stride + 1
    out = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            region = image[i * stride:i * stride + kh, j * stride:j * stride + kw]
            out[i, j] = np.sum(region * kernel)
    return out

# Kiçik nümunə: 5x5 "şəkil" və 3x3 filtr
img = np.array([[1, 2, 3, 0, 1],
                [0, 1, 2, 3, 1],
                [1, 0, 1, 2, 2],
                [2, 1, 0, 1, 0],
                [1, 2, 1, 0, 1]], dtype=float)
k = np.array([[1, 0, -1],
              [1, 0, -1],
              [1, 0, -1]], dtype=float)

print("stride=1, padding=0 →\n", conv2d_manual(img, k))
print("\nstride=2, padding=1 →\n", conv2d_manual(img, k, stride=2, padding=1))

stride=1, padding=0 →
 [[-4. -2.  2.]
 [ 0. -4.  0.]
 [ 2.  0. -1.]]

stride=2, padding=1 →
 [[-3.  0.  3.]
 [-2. -4.  6.]
 [-3.  2.  1.]]


🤔 **Sual:** düstura görə yoxlayın — 5×5 şəkil, 3×3 filtr, `stride=2`, `padding=1` olduqda çıxış niyə 3×3 olur?

In [8]:
def conv_output_size(W, K, P=0, S=1):
    return (W - K + 2 * P) // S + 1

print(conv_output_size(5, 3))            # 3
print(conv_output_size(5, 3, P=1, S=2))  # 3
print(conv_output_size(32, 3, P=1))      # 32  ← padding="same"
print(conv_output_size(32, 5))           # 28  ← LeNet-in ilk qatı

3
3
32
28


### Klassik filtrlər real şəkildə

Neyron şəbəkədən əvvəl filtrlər **əl ilə** seçilirdi. Məsələn, Sobel filtri kənarları (edges) tapır.
CNN-in əsas ideyası: **filtrlərin dəyərlərini şəbəkə özü öyrənir.**

In [9]:
gray = x_train[7].mean(axis=2)   # rəngli şəkli bozlaşdırırıq (sadəlik üçün)

filters = {
    "Şaquli kənarlar (Sobel X)": np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]),
    "Üfüqi kənarlar (Sobel Y)": np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]),
    "Kəskinləşdirmə (sharpen)": np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]]),
    "Bulanıqlaşdırma (blur)": np.ones((3, 3)) / 9,
}

plt.figure(figsize=(15, 3.5))
plt.subplot(1, 5, 1)
plt.imshow(gray, cmap="gray"); plt.title("Orijinal"); plt.axis("off")
for i, (name, f) in enumerate(filters.items()):
    plt.subplot(1, 5, i + 2)
    plt.imshow(conv2d_manual(gray, f, padding=1), cmap="gray")
    plt.title(name, fontsize=9); plt.axis("off")
plt.show()

/var/folders/9g/p_yhbqws45xfhb8zmmg367j00000gn/T/ipykernel_11587/434660743.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Bizim funksiya TensorFlow ilə eyni nəticəni verirmi?

Qeyd: dərin öyrənmə kitabxanalarında "konvolyusiya" əslində **cross-correlation**-dur (filtr çevrilmir) — biz də elə yazmışıq.

In [10]:
f = filters["Şaquli kənarlar (Sobel X)"].astype("float32")
tf_out = tf.nn.conv2d(gray[None, :, :, None].astype("float32"),   # (batch, H, W, kanal)
                      f[:, :, None, None],                        # (kH, kW, in, out)
                      strides=1, padding="SAME")[0, :, :, 0].numpy()
my_out = conv2d_manual(gray, f, padding=1)
print("Maksimal fərq:", np.abs(tf_out - my_out).max())

Maksimal fərq: 2.384185791015625e-07


### Konvolyusiya qatı (Conv2D) və pooling

Real şəkildə bir neçə kanal var (RGB). Ona görə də bir filtrin ölçüsü əslində **K × K × C_in**-dir, və qatda **C_out** sayda filtr olur.

Parametrlərin sayı:
$$
(K \cdot K \cdot C_{in} + 1) \cdot C_{out}
$$

**Pooling** qatı xüsusiyyət xəritəsini kiçildir: `MaxPooling2D(2)` hər 2×2 pəncərədən maksimumu götürür.

In [11]:
demo = keras.Sequential([
    keras.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, kernel_size=3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64, kernel_size=3, padding="same", activation="relu"),
])
demo.summary()

print("Əl ilə hesablama, 1-ci qat:", (3 * 3 * 3 + 1) * 32)
print("Əl ilə hesablama, 2-ci qat:", (3 * 3 * 32 + 1) * 64)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,392 (75.75 KB)

 Trainable params: 19,392 (75.75 KB)

 Non-trainable params: 0 (0.00 B)

Əl ilə hesablama, 1-ci qat: 896
Əl ilə hesablama, 2-ci qat: 18496


In [12]:
# MaxPooling-in kiçik nümunəsi
a = np.array([[1, 3, 2, 1],
              [4, 6, 5, 0],
              [7, 2, 9, 8],
              [1, 0, 3, 4]], dtype="float32")
pooled = layers.MaxPooling2D(2)(a[None, :, :, None])[0, :, :, 0].numpy()
print(pooled)

[[6. 5.]
 [7. 9.]]


## 3. Çoxsinifli təsnifat: Softmax və Cross-Entropy

Şəbəkənin son qatı 10 ədəd verir (**logits**). **Softmax** onları ehtimallara çevirir:

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

İtki funksiyası — **Categorical Cross-Entropy**: düzgün sinfin ehtimalının mənfi loqarifmi:

$$
L = -\log p_{\text{düzgün sinif}}
$$

Keras-da iki variant var:
- `categorical_crossentropy` — etiketlər one-hot formasında: `[0,0,1,0,...]`;
- `sparse_categorical_crossentropy` — etiketlər sadəcə rəqəmdir: `2`. Biz bunu istifadə edirik.

In [13]:
def softmax(z):
    z = z - z.max()          # ədədi stabillik üçün
    e = np.exp(z)
    return e / e.sum()

logits = np.array([2.0, 1.0, 0.1, -1.0, 0.5, 0.0, -0.5, 1.5, 0.2, -2.0])
probs = softmax(logits)
true_class = 0

print("Ehtimallar:", np.round(probs, 3))
print("Cəmi:", probs.sum())
print("Əl ilə CE itkisi:", -np.log(probs[true_class]))

keras_loss = keras.losses.SparseCategoricalCrossentropy()
print("Keras CE itkisi:", float(keras_loss(np.array([true_class]), probs[None, :])))

Ehtimallar: [0.357 0.131 0.053 0.018 0.08  0.048 0.029 0.217 0.059 0.007]
Cəmi: 1.0
Əl ilə CE itkisi: 1.0288801288297968
Keras CE itkisi: 1.0288801193237305


🤔 **Sual:** model düzgün sinfə 0.99 ehtimal versə, itki nə qədər olar? 0.01 versə? Niyə səhv və "özünə əmin" proqnoz çox böyük cəzalandırılır?

In [14]:
for p in [0.99, 0.5, 0.1, 0.01]:
    print(f"p = {p:<5} → itki = {-np.log(p):.3f}")

p = 0.99  → itki = 0.010
p = 0.5   → itki = 0.693
p = 0.1   → itki = 2.303
p = 0.01  → itki = 4.605


## 4. İlk modellər: MLP vs CNN vs LeNet-5

Bütün modelləri eyni şəkildə öyrətmək və müqayisə etmək üçün köməkçi funksiyalar yazaq.

In [15]:
results = {}      # bütün eksperimentlərin nəticələri
trained = {}      # öyrədilmiş modellər

def train_model(model, name, epochs=EPOCHS, batch_size=BATCH_SIZE, optimizer=None, verbose=2):
    if optimizer is None:
        optimizer = keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(optimizer=optimizer,
                  loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    t0 = time.time()
    history = model.fit(x_train, y_train,
                        validation_data=(x_val, y_val),
                        epochs=epochs, batch_size=batch_size, verbose=verbose)
    train_time = time.time() - t0
    test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
    results[name] = {"history": history.history, "test_acc": test_acc,
                     "params": model.count_params(), "time": train_time}
    trained[name] = model
    print(f"\n{name}: test dəqiqliyi = {test_acc:.4f}, "
          f"parametrlər = {model.count_params():,}, vaxt = {train_time:.0f} san")
    return history

def plot_histories(names, title=""):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for name in names:
        h = results[name]["history"]
        axes[0].plot(h["val_accuracy"], label=f"{name} (val)")
        axes[0].plot(h["accuracy"], "--", alpha=0.5, label=f"{name} (train)")
        axes[1].plot(h["val_loss"], label=f"{name} (val)")
    axes[0].set_title("Dəqiqlik (accuracy)"); axes[0].set_xlabel("Epoxa")
    axes[1].set_title("Doğrulama itkisi (val loss)"); axes[1].set_xlabel("Epoxa")
    axes[0].legend(fontsize=8); axes[1].legend(fontsize=8)
    fig.suptitle(title)
    plt.show()

### 4.1. Baza modeli: tam əlaqəli şəbəkə (MLP)

Şəkli düz vektora çeviririk (32·32·3 = 3072 ədəd) və adi Dense qatlarına veririk.
Problem: piksellərin **məkan strukturu** itir — şəbəkə hansı piksellərin qonşu olduğunu bilmir.

In [16]:
def build_mlp():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dense(256, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="MLP")

mlp = build_mlp()
mlp.summary()
train_model(mlp, "MLP")

Model: "MLP"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ flatten (Flatten)               │ (None, 3072)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │     1,573,376 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,707,274 (6.51 MB)

 Trainable params: 1,707,274 (6.51 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15


... (26 outputs left out)

352/352 - 2s - 6ms/step - accuracy: 0.5454 - loss: 1.2830 - val_accuracy: 0.4930 - val_loss: 1.4445


Epoch 15/15


352/352 - 2s - 6ms/step - accuracy: 0.5502 - loss: 1.2688 - val_accuracy: 0.4964 - val_loss: 1.4392



MLP: test dəqiqliyi = 0.4900, parametrlər = 1,707,274, vaxt = 32 san


### 4.2. Sadə CNN

Struktur: `[Conv → Conv → Pool] × 2 → Dense → Softmax`.
Konvolyusiya qatları lokal xüsusiyyətləri (kənarlar, teksturalar) tapır, sonrakı qatlar onları daha mürəkkəb formalara birləşdirir.

In [17]:
def build_simple_cnn():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(10, activation="softmax"),
    ], name="Simple_CNN")

simple_cnn = build_simple_cnn()
simple_cnn.summary()
train_model(simple_cnn, "Simple CNN")

Model: "Simple_CNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 32, 32, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 16, 16, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │       524,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 591,274 (2.26 MB)

 Trainable params: 591,274 (2.26 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15


... (12 outputs left out)

352/352 - 19s - 55ms/step - accuracy: 0.7228 - loss: 0.7917 - val_accuracy: 0.7502 - val_loss: 0.7243


Epoch 8/15


352/352 - 19s - 55ms/step - accuracy: 0.7392 - loss: 0.7423 - val_accuracy: 0.7520 - val_loss: 0.7377


Epoch 9/15


In [ ]:
plot_histories(["MLP", "Simple CNN"], "MLP vs CNN")

🤔 **Suallar:**
1. Hansı modeldə parametr daha çoxdur? Hansı daha yaxşı nəticə verir? Niyə?
2. Train və val əyrilərinə baxın: hansı modeldə **overfitting** (həddən artıq öyrənmə) görünür?

### 4.3. LeNet-5 (Yann LeCun, 1998)

İlk uğurlu CNN-lərdən biri — əlyazma rəqəmlərini tanımaq üçün yaradılıb. Orijinal giriş 32×32×1 idi, biz 3 kanal veririk.

| Qat | Çıxış ölçüsü |
|-----|--------------|
| Conv 5×5, 6 filtr, tanh | 28×28×6 |
| AvgPool 2×2 | 14×14×6 |
| Conv 5×5, 16 filtr, tanh | 10×10×16 |
| AvgPool 2×2 | 5×5×16 |
| Dense 120 → Dense 84 → Dense 10 | |

Xüsusiyyətləri: `tanh` aktivləşdirməsi, average pooling, padding yoxdur, cəmi ~60 min parametr.

In [ ]:
def build_lenet5():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(6, 5, activation="tanh"),
        layers.AveragePooling2D(2),
        layers.Conv2D(16, 5, activation="tanh"),
        layers.AveragePooling2D(2),
        layers.Flatten(),
        layers.Dense(120, activation="tanh"),
        layers.Dense(84, activation="tanh"),
        layers.Dense(10, activation="softmax"),
    ], name="LeNet5")

lenet = build_lenet5()
lenet.summary()
train_model(lenet, "LeNet-5")

In [ ]:
plot_histories(["MLP", "Simple CNN", "LeNet-5"], "MLP vs Simple CNN vs LeNet-5")

## 5. Batching — öyrətmə dəstinin paketlərə bölünməsi

Qradiyenti hesablamağın üç yolu var:

| Üsul | Bir addımda nümunə sayı | Xüsusiyyəti |
|------|------------------------|-------------|
| Batch Gradient Descent | bütün dəst (45 000) | dəqiq qradiyent, amma çox yavaş və yaddaşa sığmır |
| Stochastic GD | 1 | çox səs-küylü, GPU-dan səmərəsiz istifadə |
| **Mini-batch GD** | 32–512 | qızıl orta — praktikada hamı bunu istifadə edir |

Terminlər:
- **batch** — bir addımda emal olunan nümunələr qrupu;
- **iteration (step)** — bir batch üzrə çəkilərin bir dəfə yenilənməsi;
- **epoch** — bütün öyrətmə dəstinin bir dəfə keçilməsi. `steps_per_epoch = ⌈N / batch_size⌉`.

Hər epoxadan əvvəl məlumatları **qarışdırmaq (shuffle)** vacibdir — əks halda batch-lər hər dəfə eyni olur.

`model.fit(..., batch_size=128)` bunu avtomatik edir. Gəlin `tf.data` ilə əl ilə edək:

In [ ]:
batch_size = 64
train_ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
            .shuffle(10_000)                 # qarışdırma
            .batch(batch_size)               # paketlərə bölmə
            .prefetch(tf.data.AUTOTUNE))     # növbəti batch-i əvvəlcədən hazırlamaq

for xb, yb in train_ds.take(1):
    print("Bir batch:", xb.shape, yb.shape)

print("Bir epoxada addım sayı:", len(train_ds), "=", f"ceil({len(x_train)} / {batch_size})")

### Eksperiment: batch ölçüsünün təsiri

Eyni modeli müxtəlif batch ölçüləri ilə öyrədək və **vaxta** və **dəqiqliyə** baxaq.

In [ ]:
batch_results = {}
for bs in [16, 128, 1024]:
    keras.utils.set_random_seed(42)
    model = build_simple_cnn()
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    t0 = time.time()
    h = model.fit(x_train, y_train, validation_data=(x_val, y_val),
                  epochs=SHORT_EPOCHS, batch_size=bs, verbose=0)
    batch_results[bs] = {"val_acc": h.history["val_accuracy"], "time": time.time() - t0}
    print(f"batch_size={bs:5d}: addım/epoxa = {int(np.ceil(len(x_train) / bs)):5d}, "
          f"vaxt = {batch_results[bs]['time']:.0f} san, "
          f"son val_acc = {h.history['val_accuracy'][-1]:.4f}")

plt.figure(figsize=(7, 4))
for bs, r in batch_results.items():
    plt.plot(r["val_acc"], marker="o", label=f"batch={bs}")
plt.xlabel("Epoxa"); plt.ylabel("val accuracy"); plt.legend()
plt.title("Batch ölçüsünün təsiri")
plt.show()

🤔 **Suallar:**
1. Niyə kiçik batch ilə bir epoxa daha uzun çəkir?
2. Niyə böyük batch ilə eyni sayda epoxada dəqiqlik daha aşağı ola bilər? (İpucu: addımların sayı.)
3. Böyük batch istifadə etsək, learning rate-i necə dəyişmək məntiqlidir?

## 6. Dərin şəbəkələrin problemləri və Batch Normalization

### 6.1. Problem: itən qradiyentlər (vanishing gradients)

Geri yayılmada (backpropagation) qradiyent hər qatdan keçərkən törəmələrə vurulur. Qatlar çox olduqda:
- törəmələr < 1 olarsa → qradiyent eksponensial **kiçilir** (vanishing) — ilk qatlar praktiki olaraq öyrənmir;
- törəmələr > 1 olarsa → qradiyent **partlayır** (exploding) — öyrətmə qeyri-stabil olur.

Məsələn, `sigmoid`-in törəməsi ən çoxu 0.25-dir. 15 qatdan sonra: $0.25^{15} \approx 10^{-9}$.

Əlavə problem — **daxili kovariat sürüşməsi (internal covariate shift)**: hər qatın girişinin paylanması öyrətmə zamanı daim dəyişir, sonrakı qatlar "hərəkət edən hədəfə" uyğunlaşmalı olur.

Gəlin bunu öz gözümüzlə görək: 15 qatlı şəbəkədə hər qatın qradiyent normasını ölçək.

In [ ]:
def build_deep_plain(depth=15, activation="relu", use_bn=False, filters=32):
    model = keras.Sequential([keras.Input(shape=(32, 32, 3))], name=f"deep_{activation}_bn{use_bn}")
    for i in range(depth):
        model.add(layers.Conv2D(filters, 3, padding="same", use_bias=not use_bn))
        if use_bn:
            model.add(layers.BatchNormalization())
        model.add(layers.Activation(activation))
        if i in (depth // 3, 2 * depth // 3):
            model.add(layers.MaxPooling2D(2))
    model.add(layers.GlobalAveragePooling2D())
    model.add(layers.Dense(10, activation="softmax"))
    return model

def layer_gradient_norms(model, x, y):
    kernels = [l.kernel for l in model.layers if isinstance(l, (layers.Conv2D, layers.Dense))]
    loss_fn = keras.losses.SparseCategoricalCrossentropy()
    with tf.GradientTape() as tape:
        loss = loss_fn(y, model(x, training=True))
    grads = tape.gradient(loss, kernels)
    return [float(tf.norm(g)) for g in grads]

xb, yb = x_train[:256], y_train[:256]
plt.figure(figsize=(9, 4))
for act, bn in [("sigmoid", False), ("relu", False), ("relu", True)]:
    keras.utils.set_random_seed(0)
    m = build_deep_plain(activation=act, use_bn=bn)
    norms = layer_gradient_norms(m, xb, yb)
    plt.semilogy(range(1, len(norms) + 1), norms, marker="o", label=f"{act}, BN={bn}")
plt.xlabel("Qatın nömrəsi (1 = girişə ən yaxın)")
plt.ylabel("Qradiyentin norması (log şkala)")
plt.title("Dərin şəbəkədə qradiyentlərin paylanması")
plt.legend(); plt.grid(alpha=0.3)
plt.show()

🤔 **Sual:** `sigmoid` əyrisində ilk qatlarla son qatlar arasında qradiyent neçə dəfə fərqlənir? BN əlavə etdikdə nə dəyişir?

### 6.2. Batch Normalization (Ioffe & Szegedy, 2015)

Hər batch üçün, hər kanal üzrə ayrıca:

$$
\mu_B = \frac{1}{m}\sum x_i, \qquad \sigma_B^2 = \frac{1}{m}\sum (x_i - \mu_B)^2
$$
$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \varepsilon}}, \qquad y_i = \gamma \hat{x}_i + \beta
$$

- $\gamma$ və $\beta$ — **öyrənilən** parametrlərdir (şəbəkə lazım olsa normallaşdırmanı "geri qaytara" bilər);
- **öyrətmə zamanı** batch-in statistikası istifadə olunur;
- **proqnoz (inference) zamanı** — öyrətmə boyu toplanmış hərəkətli orta (moving mean / moving variance).

Gəlin BN-i əl ilə hesablayıb Keras ilə müqayisə edək:

In [ ]:
x_demo = np.random.randn(8, 4, 4, 3).astype("float32") * 5 + 10   # (batch, H, W, kanal)

# Əl ilə: hər kanal üçün (batch, H, W) oxları üzrə orta və dispersiya
mean = x_demo.mean(axis=(0, 1, 2))
var = x_demo.var(axis=(0, 1, 2))
eps = 1e-3                                       # Keras-da default epsilon
manual_bn = (x_demo - mean) / np.sqrt(var + eps)  # gamma=1, beta=0 (başlanğıc dəyərlər)

bn_layer = layers.BatchNormalization()
keras_bn = bn_layer(x_demo, training=True).numpy()

print("Girişin kanallar üzrə ortası:", np.round(mean, 2))
print("Çıxışın kanallar üzrə ortası:", np.round(keras_bn.mean(axis=(0, 1, 2)), 4))
print("Çıxışın kanallar üzrə std-si:", np.round(keras_bn.std(axis=(0, 1, 2)), 4))
print("Əl ilə və Keras arasında maksimal fərq:", np.abs(manual_bn - keras_bn).max())
print("Öyrənilən parametrlər:", [w.name for w in bn_layer.trainable_weights])
print("Öyrənilməyən (moving) parametrlər:", [w.name for w in bn_layer.non_trainable_weights])

### 6.3. Eksperiment: CNN + BatchNorm

Sadə CNN-i götürüb hər konvolyusiyadan sonra BN əlavə edirik. Standart sıra: `Conv (bias olmadan) → BN → ReLU`.

🤔 Niyə Conv-da `use_bias=False`? (İpucu: BN-dəki $\beta$ nə edir?)

In [ ]:
def conv_bn_relu(filters):
    return [layers.Conv2D(filters, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU()]

def build_cnn_bn():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        *conv_bn_relu(32), *conv_bn_relu(32), layers.MaxPooling2D(2),
        *conv_bn_relu(64), *conv_bn_relu(64), layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, use_bias=False),
        layers.BatchNormalization(),
        layers.ReLU(),
        layers.Dropout(0.5),
        layers.Dense(10, activation="softmax"),
    ], name="CNN_BN")

cnn_bn = build_cnn_bn()
train_model(cnn_bn, "CNN + BN")
plot_histories(["Simple CNN", "CNN + BN"], "BatchNorm-un təsiri")

### 6.4. (Əlavə) Dərin şəbəkə: BN ilə və BN olmadan

15 qatlı "düz" şəbəkəni hər iki variantda qısa müddət öyrədək.

In [ ]:
for bn in [False, True]:
    keras.utils.set_random_seed(42)
    m = build_deep_plain(depth=15, activation="relu", use_bn=bn)
    train_model(m, f"Deep-15 BN={bn}", epochs=SHORT_EPOCHS, verbose=0)
plot_histories(["Deep-15 BN=False", "Deep-15 BN=True"], "Dərin şəbəkə: BN ilə və BN olmadan")

## 7. CNN arxitekturaları: ImageNet və AlexNet

### ImageNet — arxitektura deyil, dataset!

**ImageNet** — 14 milyondan çox şəkildən ibarət böyük datasetdir. 2010–2017-ci illərdə onun əsasında **ILSVRC** yarışması keçirilirdi: 1.2 milyon şəkil, **1000 sinif**. Bu yarışma CNN arxitekturalarının inkişafının əsas mühərriki oldu:

| İl | Model | Top-5 səhv | Əsas yenilik |
|----|-------|-----------|--------------|
| 1998 | LeNet-5 | — (MNIST) | ilk praktik CNN |
| 2012 | **AlexNet** | 15.3% | ReLU, Dropout, GPU-da öyrətmə, dərin CNN |
| 2014 | VGG-16 | 7.3% | yalnız 3×3 filtrlər, dərinlik |
| 2015 | **ResNet-152** | 3.6% | qalıq (residual) əlaqələr, 152 qat |

(İnsanın səhvi təxminən 5% qiymətləndirilir.)

### AlexNet (Krizhevsky, Sutskever, Hinton, 2012)

Orijinal: 227×227 giriş, 5 konvolyusiya + 3 Dense qat, ~60 milyon parametr. Yenilikləri:
- **ReLU** — sigmoid/tanh-dan qat-qat sürətli öyrənir;
- **Dropout** — tam əlaqəli qatlarda overfitting-ə qarşı;
- **Data augmentation** — şəkillərin təsadüfi kəsilməsi və əks etdirilməsi;
- iki GPU-da paralel öyrətmə.

32×32 şəkillər üçün kiçildilmiş versiya qururuq (strukturu saxlayırıq, ölçüləri azaldırıq):

In [ ]:
def build_mini_alexnet():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),                                    # 16x16
        layers.Conv2D(192, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),                                    # 8x8
        layers.Conv2D(384, 3, padding="same", activation="relu"),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.Conv2D(256, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(2),                                    # 4x4
        layers.Flatten(),
        layers.Dropout(0.5),
        layers.Dense(1024, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1024, activation="relu"),
        layers.Dense(10, activation="softmax"),
    ], name="Mini_AlexNet")

alexnet = build_mini_alexnet()
alexnet.summary()
train_model(alexnet, "Mini-AlexNet", optimizer=keras.optimizers.Adam(5e-4))

In [ ]:
plot_histories(["LeNet-5", "Simple CNN", "Mini-AlexNet"], "LeNet vs AlexNet")

## 8. ResNet — qalıq (residual) əlaqələr

### Paradoks
Qatları artırdıqca şəbəkə daha güclü olmalıdır. Amma praktikada 20 qatlı "düz" şəbəkə 56 qatlıdan **öyrətmə dəstində də** daha yaxşı nəticə verir. Bu overfitting deyil — bu **optimallaşdırma problemi**dir: dərin şəbəkə hətta "heç nə etməmək" (identik funksiya) öyrənməkdə çətinlik çəkir.

### Həll (He et al., 2015)
Qat $H(x)$ funksiyasını birbaşa öyrənmək əvəzinə **qalığı** $F(x) = H(x) - x$ öyrənir:

$$
y = F(x) + x
$$

```
x ──► [Conv → BN → ReLU → Conv → BN] ──► (+) ──► ReLU ──► y
│                                         ▲
└──────────── shortcut (qısa yol) ────────┘
```

Faydaları:
- $F(x) = 0$ öyrənmək asandır → qat lazım deyilsə, sadəcə $x$-i ötürür;
- qradiyent "qısa yol" ilə birbaşa ilk qatlara çatır → itən qradiyent problemi zəifləyir.

Ölçülər fərqlənəndə (stride=2 və ya filtr sayı dəyişəndə) shortcut-da 1×1 konvolyusiya istifadə olunur.

Bu dəfə **funksional API** istifadə edirik — çünki Sequential "budaqlanan" strukturu qura bilmir.
Həmçinin **data augmentation** əlavə edirik (AlexNet-dəki kimi).

In [ ]:
def residual_block(x, filters, stride=1):
    shortcut = x
    y = layers.Conv2D(filters, 3, strides=stride, padding="same", use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.ReLU()(y)
    y = layers.Conv2D(filters, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)

    # Ölçülər uyğun gəlmirsə, shortcut-u 1x1 konvolyusiya ilə uyğunlaşdırırıq
    if stride != 1 or shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, strides=stride, use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)

    y = layers.Add()([y, shortcut])     # ← əsas ideya: F(x) + x
    return layers.ReLU()(y)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomTranslation(0.1, 0.1),
], name="augmentation")

def build_mini_resnet(augment=True):
    inputs = keras.Input(shape=(32, 32, 3))
    x = data_augmentation(inputs) if augment else inputs
    x = layers.Conv2D(32, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    for filters, stride in [(32, 1), (32, 1), (64, 2), (64, 1), (128, 2), (128, 1)]:
        x = residual_block(x, filters, stride)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation="softmax")(x)
    return keras.Model(inputs, outputs, name="Mini_ResNet")

resnet = build_mini_resnet()
resnet.summary()

In [ ]:
# Augmentation-un nəticəsinə baxaq
plt.figure(figsize=(12, 2.5))
for i in range(8):
    aug = data_augmentation(x_train[:1], training=True)[0].numpy()
    plt.subplot(1, 8, i + 1); plt.imshow(np.clip(aug, 0, 1)); plt.axis("off")
plt.suptitle("Bir şəklin augmentation variantları")
plt.show()

In [ ]:
train_model(resnet, "Mini-ResNet", epochs=EPOCHS + 5)
plot_histories(["CNN + BN", "Mini-AlexNet", "Mini-ResNet"], "ResNet vs digər modellər")

🤔 **Sual:** ResNet-də Flatten + böyük Dense qatları əvəzinə `GlobalAveragePooling2D` istifadə etdik. Bu parametrlərin sayına necə təsir edir? `summary()` ilə AlexNet-i müqayisə edin.

### (Bonus) ImageNet-də öyrədilmiş real ResNet-50

Keras-da ImageNet üzərində öyrədilmiş hazır modellər var. Gəlin real ResNet-50-ni yükləyib CIFAR şəkillərini ona göstərək (şəkilləri 224×224-ə böyüdürük).
Qeyd: ImageNet-in 1000 sinfi var, nəticələr CIFAR siniflərindən daha detallı olacaq.

In [ ]:
resnet50 = keras.applications.ResNet50(weights="imagenet")
print(f"ResNet-50 parametrləri: {resnet50.count_params():,}, qat sayı: {len(resnet50.layers)}")

idx = [3, 6, 8, 12]   # bir neçə test şəkli
imgs = tf.image.resize(x_test[idx] * 255.0, (224, 224))
preds = resnet50.predict(keras.applications.resnet50.preprocess_input(imgs), verbose=0)
decoded = keras.applications.resnet50.decode_predictions(preds, top=1)

plt.figure(figsize=(12, 3))
for i, j in enumerate(idx):
    plt.subplot(1, 4, i + 1)
    plt.imshow(x_test[j]); plt.axis("off")
    plt.title(f"Həqiqi: {class_names[y_test[j]]}\nResNet50: {decoded[i][0][1]}", fontsize=9)
plt.show()

## 9. Qradiyentin normallaşdırılması və Gradient Clipping

Partlayan qradiyentlər (exploding gradients) çəkilərdə nəhəng addımlara və `loss = NaN`-a gətirib çıxarır. İki yaxın texnika:

**Gradient clipping (norma üzrə kəsmə)** — qradiyentin norması həddi aşdıqda onu kiçildirik, istiqaməti saxlayırıq:
$$
g \leftarrow g \cdot \min\left(1, \frac{\tau}{\lVert g \rVert}\right)
$$

**Qradiyentin normallaşdırılması** — qradiyenti həmişə vahid normaya gətiririk, addımın ölçüsünü yalnız learning rate təyin edir:
$$
g \leftarrow \frac{g}{\lVert g \rVert + \varepsilon}
$$

Keras-da hazır variantlar (optimizer parametrləri):
- `clipnorm=1.0` — **hər** çəki tensoru ayrıca kəsilir;
- `global_clipnorm=1.0` — **bütün** qradiyentlərin ümumi norması kəsilir (ən çox istifadə olunan);
- `clipvalue=0.5` — hər element $[-0.5, 0.5]$ aralığına sıxılır (istiqaməti dəyişə bilər!).

Prosesi aydın görmək üçün **öz öyrətmə dövrümüzü (custom training loop)** yazaq və hər addımda qradiyentin normasını qeyd edək.

In [ ]:
def train_custom(model, lr, grad_mode=None, clip_value=1.0, epochs=2, batch_size=128):
    optimizer = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)
    loss_fn = keras.losses.SparseCategoricalCrossentropy()
    ds = (tf.data.Dataset.from_tensor_slices((x_train, y_train))
          .shuffle(10_000).batch(batch_size).prefetch(tf.data.AUTOTUNE))

    @tf.function
    def train_step(xb, yb):
        with tf.GradientTape() as tape:
            loss = loss_fn(yb, model(xb, training=True))
        grads = tape.gradient(loss, model.trainable_variables)
        global_norm = tf.linalg.global_norm(grads)          # kəsmədən ƏVVƏLKİ norma

        if grad_mode == "clip":
            grads, _ = tf.clip_by_global_norm(grads, clip_value)
        elif grad_mode == "normalize":
            grads = [g / (global_norm + 1e-6) for g in grads]

        optimizer.apply_gradients(zip(grads, model.trainable_variables))
        return loss, global_norm

    loss_log, norm_log = [], []
    for epoch in range(epochs):
        for xb, yb in ds:
            loss, gnorm = train_step(xb, yb)
            loss_log.append(float(loss)); norm_log.append(float(gnorm))
    model.compile(loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    _, val_acc = model.evaluate(x_val, y_val, verbose=0)
    return loss_log, norm_log, val_acc

### Eksperiment

BN olmadan 10 qatlı şəbəkə + **böyük learning rate** — qeyri-stabil öyrətmə üçün ideal şərait. Üç variantı müqayisə edirik.

Əgər sizdə "partlayış" görünmürsə, `LR`-i artırın (0.1, 0.2).

In [ ]:
LR = 0.05
clip_logs = {}
for mode in [None, "clip", "normalize"]:
    keras.utils.set_random_seed(42)
    m = build_deep_plain(depth=10, activation="relu", use_bn=False)
    loss_log, norm_log, val_acc = train_custom(m, lr=LR, grad_mode=mode, clip_value=1.0)
    clip_logs[str(mode)] = (loss_log, norm_log)
    last_loss = np.nanmean(loss_log[-50:])
    print(f"grad_mode={str(mode):10s}: son itki ≈ {last_loss:.3f}, val_acc = {val_acc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for mode, (loss_log, norm_log) in clip_logs.items():
    axes[0].plot(loss_log, alpha=0.7, label=mode)
    axes[1].semilogy(norm_log, alpha=0.7, label=mode)
axes[0].set_title("Hər addımda itki"); axes[0].set_xlabel("Addım"); axes[0].set_ylim(0, 5)
axes[1].set_title("Qradiyentin norması (kəsmədən əvvəl)"); axes[1].set_xlabel("Addım")
axes[0].legend(); axes[1].legend()
plt.show()

🤔 **Suallar:**
1. Clipping olmadan itki əyrisində nə görürsünüz? Sıçrayışlar varmı, `NaN` yarandımı?
2. `clip` və `normalize` arasında fərq nədir? Normallaşdırmada qradiyent kiçik olduqda nə baş verir (məs. minimuma yaxın)?
3. Clipping-in həddini (`clip_value`) necə seçmək olar? (İpucu: normanın qrafikinə baxın.)

In [ ]:
# Keras-da hazır variant — bir sətirlə
keras.utils.set_random_seed(42)
m = build_simple_cnn()
train_model(m, "Simple CNN + clipnorm",
            optimizer=keras.optimizers.Adam(1e-3, global_clipnorm=1.0),
            epochs=SHORT_EPOCHS, verbose=0)

## 10. Layer Normalization (LayerNorm)

BatchNorm-un zəif cəhətləri:
- **batch ölçüsündən asılıdır** — kiçik batch-də (məs. 2–4) statistika səs-küylü olur;
- öyrətmə və proqnoz zamanı fərqli işləyir (moving statistics);
- ardıcıllıqlar (RNN, Transformer) üçün yaxşı uyğun gəlmir.

**LayerNorm (Ba et al., 2016)** normallaşdırmanı **batch üzrə deyil, hər nümunənin daxilində** aparır:

| | Hansı oxlar üzrə orta/dispersiya | Nəyə görə |
|--|------|------|
| BatchNorm | (N, H, W) — batch və məkan | hər kanal üçün ayrıca |
| LayerNorm | (H, W, C) — bir nümunənin içində | hər nümunə üçün ayrıca |

LayerNorm Transformer arxitekturalarının (GPT, BERT, ViT) standart komponentidir.

In [ ]:
x_demo = np.random.randn(4, 8, 8, 16).astype("float32") * 3 + 2   # (N, H, W, C)

ln = layers.LayerNormalization(axis=[1, 2, 3])   # hər nümunə üçün bütün (H, W, C) üzrə
bn = layers.BatchNormalization()

ln_out = ln(x_demo).numpy()
bn_out = bn(x_demo, training=True).numpy()

print("LayerNorm: hər NÜMUNƏNİN ortası:", np.round(ln_out.mean(axis=(1, 2, 3)), 4))
print("BatchNorm: hər KANALIN ortası (ilk 5):", np.round(bn_out.mean(axis=(0, 1, 2))[:5], 4))

# Əl ilə LayerNorm
mu = x_demo.mean(axis=(1, 2, 3), keepdims=True)
var = x_demo.var(axis=(1, 2, 3), keepdims=True)
manual_ln = (x_demo - mu) / np.sqrt(var + ln.epsilon)
print("Əl ilə LN və Keras arasında fərq:", np.abs(manual_ln - ln_out).max())

### Batch-dən asılılıq: maraqlı eksperiment

Eyni şəkli iki **fərqli** batch-in içində normallaşdıraq. LayerNorm-un nəticəsi dəyişəcəkmi? BatchNorm-unku?

In [ ]:
sample = x_demo[:1]
batch_a = np.concatenate([sample, x_demo[1:2]])
batch_b = np.concatenate([sample, x_demo[2:3] * 10])   # tamamilə başqa "qonşu"

bn_a = bn(batch_a, training=True).numpy()[0]
bn_b = bn(batch_b, training=True).numpy()[0]
ln_a = ln(batch_a).numpy()[0]
ln_b = ln(batch_b).numpy()[0]

print("BatchNorm: eyni şəklin çıxışları arasındakı fərq =", np.abs(bn_a - bn_b).max())
print("LayerNorm: eyni şəklin çıxışları arasındakı fərq =", np.abs(ln_a - ln_b).max())

### Eksperiment: CNN + LayerNorm vs CNN + BatchNorm

In [ ]:
def conv_ln_relu(filters):
    return [layers.Conv2D(filters, 3, padding="same"),
            layers.LayerNormalization(axis=[1, 2, 3]),
            layers.ReLU()]

def build_cnn_ln():
    return keras.Sequential([
        keras.Input(shape=(32, 32, 3)),
        *conv_ln_relu(32), *conv_ln_relu(32), layers.MaxPooling2D(2),
        *conv_ln_relu(64), *conv_ln_relu(64), layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128),
        layers.LayerNormalization(),
        layers.ReLU(),
        layers.Dropout(0.5),
        layers.Dense(10, activation="softmax"),
    ], name="CNN_LN")

cnn_ln = build_cnn_ln()
train_model(cnn_ln, "CNN + LN")
plot_histories(["Simple CNN", "CNN + BN", "CNN + LN"], "Normallaşdırma yoxdur vs BN vs LN")

## 11. Bütün modellərin müqayisəsi

In [ ]:
summary = pd.DataFrame([
    {"Model": name,
     "Test dəqiqliyi": r["test_acc"],
     "Parametrlər": r["params"],
     "Vaxt (san)": round(r["time"]),
     "Epoxa": len(r["history"]["accuracy"])}
    for name, r in results.items()
]).sort_values("Test dəqiqliyi", ascending=False).reset_index(drop=True)
summary.style.format({"Test dəqiqliyi": "{:.4f}", "Parametrlər": "{:,}"})

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(summary["Model"][::-1], summary["Test dəqiqliyi"][::-1])
for i, v in enumerate(summary["Test dəqiqliyi"][::-1]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center")
ax.set_xlabel("Test dəqiqliyi"); ax.set_xlim(0, 1)
ax.set_title("Modellərin müqayisəsi (CIFAR-10)")
plt.show()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(summary["Parametrlər"], summary["Test dəqiqliyi"])
for _, row in summary.iterrows():
    ax.annotate(row["Model"], (row["Parametrlər"], row["Test dəqiqliyi"]), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("Parametrlərin sayı (log)"); ax.set_ylabel("Test dəqiqliyi")
ax.set_title("Daha çox parametr = daha yaxşı model?")
plt.show()

## 12. Ən yaxşı modelin təhlili

### 12.1. Qarışıqlıq matrisi (confusion matrix)
Hansı sinifləri model bir-biri ilə qarışdırır?

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

best_name = summary.loc[0, "Model"]
best_model = trained[best_name]
print("Ən yaxşı model:", best_name)

y_pred = best_model.predict(x_test, verbose=0).argmax(axis=1)
cm = confusion_matrix(y_test, y_pred, normalize="true")

fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", values_format=".2f", xticks_rotation=45, colorbar=False)
ax.set_title(f"{best_name}: qarışıqlıq matrisi")
plt.show()

per_class = cm.diagonal()
for i in np.argsort(per_class):
    print(f"{class_names[i]:12s}: {per_class[i]:.3f}")

🤔 **Sual:** Hansı sinif cütləri ən çox qarışdırılır? Bu sizə məntiqli görünürmü?

In [ ]:
# 12.2. Səhv təsnif olunmuş şəkillər
probs = best_model.predict(x_test, verbose=0)
wrong = np.where(y_pred != y_test)[0]
# model özünə ən çox əmin olduğu səhvlər
wrong = wrong[np.argsort(-probs[wrong].max(axis=1))][:12]

plt.figure(figsize=(14, 5))
for i, j in enumerate(wrong):
    plt.subplot(2, 6, i + 1)
    plt.imshow(x_test[j]); plt.axis("off")
    plt.title(f"Həqiqi: {class_names[y_test[j]]}\nProqnoz: {class_names[y_pred[j]]} "
              f"({probs[j].max():.2f})", fontsize=8)
plt.suptitle("Modelin ən 'əmin' olduğu səhvlər")
plt.show()

### 12.3. Şəbəkə nə görür? Filtrlər və xüsusiyyət xəritələri

Sadə CNN-in birinci qatının öyrəndiyi filtrlərə baxaq. Onlar Bölmə 2-dəki əl ilə yazılmış filtrlərə (kənar aşkarlayıcılar, rəng detektorları) oxşayırmı?

In [ ]:
model_vis = trained["Simple CNN"]
first_conv = model_vis.layers[0]
w = first_conv.get_weights()[0]                 # (3, 3, 3, 32)
w = (w - w.min()) / (w.max() - w.min())         # göstərmək üçün [0, 1]-ə salırıq

plt.figure(figsize=(12, 3))
for i in range(32):
    plt.subplot(2, 16, i + 1)
    plt.imshow(w[:, :, :, i]); plt.axis("off")
plt.suptitle("Birinci konvolyusiya qatının 32 filtri (3×3×3)")
plt.show()

In [ ]:
conv_layers = [l for l in model_vis.layers if isinstance(l, layers.Conv2D)]
feature_extractor = keras.Model(inputs=model_vis.inputs,
                                outputs=[conv_layers[0].output, conv_layers[-1].output])
img_idx = 7
fmap_first, fmap_last = feature_extractor(x_test[img_idx:img_idx + 1])

plt.figure(figsize=(3, 3)); plt.imshow(x_test[img_idx]); plt.axis("off")
plt.title(class_names[y_test[img_idx]]); plt.show()

for fmap, title in [(fmap_first, "1-ci konvolyusiya qatı (32×32)"),
                    (fmap_last, "Sonuncu konvolyusiya qatı (16×16)")]:
    plt.figure(figsize=(14, 2.5))
    for i in range(16):
        plt.subplot(1, 16, i + 1)
        plt.imshow(fmap[0, :, :, i], cmap="viridis"); plt.axis("off")
    plt.suptitle(title)
    plt.show()

🤔 **Sual:** birinci və sonuncu qatların xüsusiyyət xəritələri arasında nə fərq var? Hansı daha "abstrakt"dır?

## 13. Müstəqil tapşırıqlar

1. **Konvolyusiya.** `conv2d_manual` funksiyasını 3 kanallı şəkillər və bir neçə filtr üçün genişləndirin. Nəticəni `layers.Conv2D` ilə yoxlayın (çəkiləri `set_weights` ilə verin).
2. **LeNet-5.** `tanh`-ı `relu`-ya, `AveragePooling`-i `MaxPooling`-ə dəyişin. Nəticə necə dəyişdi? Niyə?
3. **Batching.** `batch_size=1024` ilə learning rate-i 4–8 dəfə artırın. Kiçik batch-in nəticəsinə yaxınlaşa bildinizmi?
4. **BatchNorm vs LayerNorm.** Hər iki modeli `batch_size=4` ilə 2 epoxa öyrədin. Hansı daha yaxşı dözür?
5. **ResNet.** Mini-ResNet-dən shortcut əlaqələrini çıxarın (`layers.Add` olmadan) və qat sayını 2 dəfə artırın. Residual və "düz" versiyanı müqayisə edin.
6. **Augmentation.** `build_mini_resnet(augment=False)` ilə öyrədin. Train və val əyriləri arasındakı fərq necə dəyişdi?
7. **Gradient clipping.** `clip_value` üçün 0.1, 1, 10 dəyərlərini sınayın. Çox kiçik hədd nəyə gətirir?
8. **Yarış!** CIFAR-10 test dəstində **≥ 85%** dəqiqliyə çatan model qurun. İstənilən texnikadan istifadə edin (daha çox epoxa, augmentation, learning rate schedule, daha dərin ResNet, transfer learning).

### Yoxlama sualları
- Konvolyusiya qatı tam əlaqəli qatdan niyə daha az parametrə malikdir?
- Softmax-ın çıxışlarının cəmi niyə həmişə 1-dir?
- BatchNorm öyrətmə və proqnoz rejimində necə fərqli işləyir?
- ResNet itən qradiyent problemini necə həll edir?
- ImageNet nədir və CNN-lərin tarixində niyə bu qədər vacibdir?
- Gradient clipping ilə qradiyentin normallaşdırılması arasında fərq nədir?
- Niyə Transformer-lərdə BatchNorm əvəzinə LayerNorm istifadə olunur?

## 14. Modelləri yadda saxlayırıq (app.py üçün)

`trained` lüğətindəki **bütün** modelləri `models/` qovluğuna `.keras` formatında yazırıq. Yanında `models.json` faylı da yaranır: modelin adı, faylı, test dəqiqliyi və parametrlərin sayı.

`app.py` bu qovluğu oxuyur və modelləri siyahıda göstərir. Siyahıdan istədiyiniz modeli seçib proqnoza baxa bilərsiniz.

In [ ]:
import json
import re
from pathlib import Path

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

index = []
for name, model in trained.items():
    # "CNN + BN" → "cnn_bn.keras", "Deep-15 BN=True" → "deep_15_bn_true.keras"
    fname = re.sub(r"[^0-9A-Za-z]+", "_", name).strip("_").lower() + ".keras"
    model.save(MODELS_DIR / fname)
    index.append({"name": name, "file": fname,
                  "test_acc": round(float(results[name]["test_acc"]), 4),
                  "params": int(results[name]["params"])})
    print(f"{name:22s} → models/{fname}")

index.sort(key=lambda m: m["test_acc"], reverse=True)   # ən yaxşı model siyahıda birinci olsun
(MODELS_DIR / "models.json").write_text(json.dumps(index, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n{len(index)} model saxlanıldı, ən yaxşısı: {index[0]['name']} ({index[0]['test_acc']:.4f})")

In [ ]:
# Yoxlayaq: diskdən yüklənən modellər eyni dəqiqliyi verirmi?
for m in index:
    loaded = keras.models.load_model(MODELS_DIR / m["file"])
    _, acc = loaded.evaluate(x_test, y_test, verbose=0)
    print(f"{m['name']:22s}: yaddaşda {m['test_acc']:.4f}, diskdən {acc:.4f}")

In [ ]:
# Colab-da işləyirsinizsə: qovluğu zip edib kompüterə yükləyin,
# sonra arxivi açıb "models" qovluğunu app.py-ın yanına qoyun.
# import shutil
# from google.colab import files
# shutil.make_archive("models", "zip", MODELS_DIR)
# files.download("models.zip")